In [2]:
import json
import requests
from urllib.parse import urljoin
import re
import shutil
import sys
from packaging import version
import pkg_resources
import platform

class InteroperabilityChecker:
    def __init__(self, json_file):
        self.repositories = self.load_metadata(json_file)

    def load_metadata(self, json_file):
        with open(json_file, 'r') as file:
            data = json.load(file)
        return {artifact_id: artifact['uri'] for artifact_id, artifact in data['artifacts'].items()}

    def get_default_branch(self, user, repo):
        api_url = f"https://api.github.com/repos/{user}/{repo}"
        try:
            response = requests.get(api_url, timeout=10)
            if response.status_code == 200:
                return response.json().get("default_branch", "main")
        except requests.RequestException:
            pass
        return "main"

    def transform_to_raw_github(self, uri, filename):
        if "github.com" in uri:
            parts = uri.strip("/").split("/")
            if len(parts) >= 5:
                user, repo = parts[3], parts[4]
                branch = self.get_default_branch(user, repo)
                return f"https://raw.githubusercontent.com/{user}/{repo}/{branch}/{filename}"
        return urljoin(uri + '/', filename)

    def check_file_exists(self, base_uri, filename):
        raw_uri = self.transform_to_raw_github(base_uri, filename)
        try:
            response = requests.head(raw_uri, timeout=10)
            return response.status_code == 200
        except requests.RequestException:
            return False

    def check_contains_filetype(self, base_uri, pattern):
        if "github.com" not in base_uri:
            return False
        parts = base_uri.strip("/").split("/")
        if len(parts) < 5:
            return False
        user, repo = parts[3], parts[4]
        branch = self.get_default_branch(user, repo)
        api_url = f"https://api.github.com/repos/{user}/{repo}/git/trees/{branch}?recursive=1"
        try:
            response = requests.get(api_url, timeout=10)
            if response.status_code == 200:
                tree = response.json().get('tree', [])
                for item in tree:
                    if re.search(pattern, item['path'], re.IGNORECASE):
                        return True
        except requests.RequestException:
            pass
        return False

    def check_python_interpreter_invocation(self, base_uri):
        patterns_to_check = [
            r"\.github/workflows/.*\.ya?ml$",
            r"Makefile",
            r".*\.sh$"
        ]
        if "github.com" not in base_uri:
            return False
        parts = base_uri.strip("/").split("/")
        if len(parts) < 5:
            return False
        user, repo = parts[3], parts[4]
        branch = self.get_default_branch(user, repo)
        api_url = f"https://api.github.com/repos/{user}/{repo}/git/trees/{branch}?recursive=1"
        try:
            response = requests.get(api_url, timeout=10)
            if response.status_code == 200:
                tree = response.json().get('tree', [])
                for item in tree:
                    path = item['path']
                    if any(re.search(p, path, re.IGNORECASE) for p in patterns_to_check):
                        raw_url = f"https://raw.githubusercontent.com/{user}/{repo}/{branch}/{path}"
                        content_resp = requests.get(raw_url, timeout=10)
                        if content_resp.status_code == 200:
                            content = content_resp.text
                            if re.search(r"\bpython[0-9.]*\b", content):
                                return True
        except requests.RequestException:
            pass
        return False

    def check_local_python_interpreter(self):
        python_path = shutil.which("python") or shutil.which("python3")
        if python_path:
            os_name = platform.system()
            print(f"✅ Python found at: {python_path} ({os_name})")
            return True
        return False

    def get_local_python_version(self):
        return f"{sys.version_info.major}.{sys.version_info.minor}"

    def extract_required_python_version(self, base_uri):
        files_to_check = ["pyproject.toml", "setup.py", "runtime.txt", ".python-version"]
        version_pattern = re.compile(r"(python_requires\s*=\s*['\"]|python\s*>=?|python-)(\d+\.\d+)")

        for file in files_to_check:
            if not self.check_file_exists(base_uri, file):
                continue
            raw_url = self.transform_to_raw_github(base_uri, file)
            try:
                response = requests.get(raw_url, timeout=10)
                if response.status_code == 200:
                    text = response.text
                    match = version_pattern.search(text)
                    if match:
                        return match.group(2)
            except requests.RequestException:
                pass
        return None

    def is_python_version_compatible(self, required_version, local_version):
        try:
            return version.parse(local_version) >= version.parse(required_version)
        except Exception:
            return False

    def check_local_requirements(self, base_uri):
        parts = base_uri.strip("/").split("/")
        if len(parts) < 5:
            return "Invalid repo URI"
        user, repo = parts[3], parts[4]
        branch = self.get_default_branch(user, repo)
        raw_url = f"https://raw.githubusercontent.com/{user}/{repo}/{branch}/requirements.txt"

        try:
            response = requests.get(raw_url, timeout=10)
            if response.status_code != 200:
                return "requirements.txt not found"

            requirements_text = response.text
            installed = []
            missing = []

            for line in requirements_text.splitlines():
                line = line.strip()
                if not line or line.startswith("#"):
                    continue
                try:
                    pkg_resources.require([line])
                    installed.append(f"{line} ✅")
                except pkg_resources.DistributionNotFound:
                    missing.append(f"{line} ❌ Not installed")
                except pkg_resources.VersionConflict as e:
                    missing.append(f"{line} ❌ Version conflict: {e.report()}")

            if not installed and not missing:
                return "requirements.txt is empty or has only comments"

            return "\n  " + "\n  ".join(installed + missing)

        except requests.RequestException:
            return "Error checking requirements"

    def check_interoperability(self):
        local_version = self.get_local_python_version()
        local_python = self.check_local_python_interpreter()
        if not local_python:
            print("❌ Python interpreter not found in PATH.")
        else:
            print(f"🖥️ Local Python Version: {local_version}")

        for artifact_id, uri in self.repositories.items():
            required_version = self.extract_required_python_version(uri)
            version_compatible = self.is_python_version_compatible(required_version, local_version) if required_version else True

            results = {
                "Python interpreter invoked in repo": self.check_python_interpreter_invocation(uri),
                "requirements.txt": self.check_file_exists(uri, "requirements.txt"),
                "All requirements installed locally": self.check_local_requirements(uri),
                "Python version specified": required_version if required_version else "Not specified",
                "Version compatible with local": "✅" if version_compatible else "❌",
            }

            print(f"\n🔧 Interoperability Check for {artifact_id} ({uri}):")
            for key, val in results.items():
                print(f" - {key}: {val}")

if __name__ == "__main__":
    json_file_path = 'artifacts.json'
    checker = InteroperabilityChecker(json_file_path)
    checker.check_interoperability()


✅ Python found at: C:\Users\irdin\AppData\Local\Microsoft\WindowsApps\python.EXE (Windows)
🖥️ Local Python Version: 3.12

🔧 Interoperability Check for artifact_1 (https://github.com/hihey54/hicss58):
 - Python interpreter invoked in repo: False
 - requirements.txt: True
 - All requirements installed locally: 
  requests>=2.31.0 ✅
  beautifulsoup4>=4.12.3 ✅
  PyMuPDF>=1.23.20 ❌ Not installed
 - Python version specified: Not specified
 - Version compatible with local: ✅

🔧 Interoperability Check for artifact_2 (https://github.com/githubkilroy/TCN-Dataset):
 - Python interpreter invoked in repo: False
 - requirements.txt: False
 - All requirements installed locally: requirements.txt not found
 - Python version specified: Not specified
 - Version compatible with local: ✅

🔧 Interoperability Check for artifact_3 (https://github.com/TheAlgorithms/Python):
 - Python interpreter invoked in repo: True
 - requirements.txt: True
 - All requirements installed locally: 
  beautifulsoup4 ✅
  httpx ✅